# 🎙️ Somali ASR (Speech-to-Text) — Colab API Server

This notebook loads the Somali wav2vec2 ASR model and serves it as a FastAPI server exposed via `ngrok`.

## Steps:
1. Run **Cell 1** to install dependencies.
2. Run **Cell 2** to check GPU.
3. Add your `NGROK_TOKEN` and `NGROK_DOMAIN` secrets in the 🔑 key icon on the left sidebar.
   - Get `NGROK_TOKEN` from [dashboard.ngrok.com/authtokens](https://dashboard.ngrok.com/authtokens)
   - Get a free static `NGROK_DOMAIN` from [dashboard.ngrok.com/domains](https://dashboard.ngrok.com/domains)
4. Run **Cell 3** to load the ASR model.
5. Run **Cell 4** to start the server and get your public URL.
6. Paste the URL into your backend `.env` as `SOMALI_ASR_URL=<url>`.

In [ ]:
# CELL 1: Install dependencies
!pip install -q transformers torch torchaudio fastapi uvicorn pyngrok python-multipart soundfile librosa
print('Dependencies installed!')

In [ ]:
# CELL 2: Verify GPU
import torch

if not torch.cuda.is_available():
    print('WARNING: No GPU found. ASR will run on CPU (slower). Consider enabling GPU.')
else:
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# CELL 3: Load Somali ASR model
import torch
from transformers import AutoModelForCTC, AutoProcessor

device = 'cuda' if torch.cuda.is_available() else 'cpu'
ASR_MODEL_ID = 'skydheere/wav2vec2-large-mms-1b-somalia'

print(f'Loading Somali ASR model on {device}...')
asr_processor = AutoProcessor.from_pretrained(ASR_MODEL_ID)
asr_model = AutoModelForCTC.from_pretrained(ASR_MODEL_ID).to(device)
asr_model.eval()
print('Somali ASR model ready!')

In [ ]:
# CELL 4: Start FastAPI server + ngrok tunnel
import io
import numpy as np
import soundfile as sf
import torch
import uvicorn
from threading import Thread
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import JSONResponse
from pyngrok import ngrok, conf
from google.colab import userdata

# Setup ngrok
NGROK_TOKEN = userdata.get('NGROK_TOKEN')
NGROK_DOMAIN = userdata.get('NGROK_DOMAIN')  # claim free static domain at dashboard.ngrok.com/domains
conf.get_default().auth_token = NGROK_TOKEN

app = FastAPI(title='Somali ASR API')

TARGET_SAMPLE_RATE = 16000

def load_audio_bytes(audio_bytes: bytes) -> np.ndarray:
    audio_io = io.BytesIO(audio_bytes)
    try:
        import librosa
        audio, sr = librosa.load(audio_io, sr=TARGET_SAMPLE_RATE, mono=True)
        return audio.astype(np.float32)
    except Exception:
        audio_io.seek(0)
        data, sr = sf.read(audio_io)
        if data.ndim > 1:
            data = data.mean(axis=1)
        if sr != TARGET_SAMPLE_RATE:
            from scipy.signal import resample_poly
            import math
            gcd = math.gcd(TARGET_SAMPLE_RATE, sr)
            data = resample_poly(data, TARGET_SAMPLE_RATE // gcd, sr // gcd)
        return data.astype(np.float32)

def transcribe_audio(audio_array: np.ndarray) -> str:
    inputs = asr_processor(
        audio_array,
        sampling_rate=TARGET_SAMPLE_RATE,
        return_tensors='pt'
    ).to(device)
    with torch.no_grad():
        logits = asr_model(**inputs).logits
    predicted_ids = torch.argmax(logits, dim=-1)
    transcription = asr_processor.decode(predicted_ids[0])
    return transcription.strip()

@app.get('/health')
def health():
    return {'status': 'online', 'model': 'skydheere/wav2vec2-large-mms-1b-somalia', 'provider': 'colab'}

@app.post('/transcribe')
async def transcribe(file: UploadFile = File(...)):
    if not file:
        raise HTTPException(status_code=400, detail='No audio file provided')
    try:
        audio_bytes = await file.read()
        audio_array = load_audio_bytes(audio_bytes)
        transcription = transcribe_audio(audio_array)
        return JSONResponse({'transcription': transcription, 'language': 'so-SO'})
    except Exception as e:
        raise HTTPException(status_code=500, detail=f'Transcription failed: {str(e)}')

# Start uvicorn in background thread
def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8001, log_level='info')

server_thread = Thread(target=run_server, daemon=True)
server_thread.start()

# Connect ngrok tunnel with static domain
tunnel = ngrok.connect(8001, 'http', domain=NGROK_DOMAIN)
public_url = tunnel.public_url

print(f'\n==========================================')
print(f'YOUR COLAB ASR URL IS READY!')
print(f'URL: {public_url}')
print(f'==========================================')
print(f'Paste this into your backend .env file:')
print(f'SOMALI_ASR_URL={public_url}')
print(f'==========================================')